# Code Snippets & Quick Reference## St. Paul Neighborhood Health ProjectReady-to-use code examples for common analysis tasks. Copy and adapt to your data.

## Setup & Libraries

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.preprocessing import MinMaxScalerfrom sklearn.cluster import KMeansimport warningswarnings.filterwarnings('ignore')print('✓ Libraries imported')# Set stylesns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (12, 6)

## Loading & Basic Exploration

In [ ]:
# Load cleaned dataperms = pd.read_csv('data/Approved_Building_Permits_-7890413957898939046.csv')crime = pd.read_csv('data/Crime_Incident_Report.csv')requests = pd.read_csv('data/Resident_Service_Requests_7024824928576740068.csv')print(f'Permits: {len(perms):,} rows')print(f'Crime: {len(crime):,} rows')print(f'Requests: {len(requests):,} rows')# Quick peek at dataprint('\nPermits sample:')perms.head(2)

In [ ]:
# Check columnsprint('Available columns in permits:')print(perms.columns.tolist())print('\nData types:')print(perms.dtypes)print('\nMissing values:')print(perms.isnull().sum())

## Neighborhood Aggregation

In [ ]:
# Crime aggregation by neighborhood and yearcrime_agg = crime.groupby(['NEIGHBORHOOD_STANDARD', 'YEAR']).agg({    'INCIDENT_ID': 'count',    'OFFENSE': 'nunique',}).rename(columns={    'INCIDENT_ID': 'TOTAL_CRIMES',    'OFFENSE': 'UNIQUE_OFFENSE_TYPES'})print('Crime aggregation:')print(crime_agg.head())print(f'\nShape: {crime_agg.shape}')# Reset index to have neighborhood and year as columnscrime_agg = crime_agg.reset_index()print(crime_agg.head())

In [ ]:
# Permits aggregationperms_agg = perms.groupby(['NEIGHBORHOOD_STANDARD', 'YEAR']).agg({    'PERMIT_ID': 'count',    'ESTIMATED_COST': ['sum', 'mean'],}).rename(columns={    'PERMIT_ID': 'TOTAL_PERMITS'})perms_agg = perms_agg.reset_index()perms_agg.columns = ['NEIGHBORHOOD_STANDARD', 'YEAR', 'TOTAL_PERMITS',                       'TOTAL_PERMIT_COST', 'AVG_PERMIT_COST']print('Permits aggregation:')print(perms_agg.head())

In [ ]:
# Service requests aggregationreqs_agg = requests.groupby(['NEIGHBORHOOD_STANDARD', 'YEAR']).agg({    'REQUEST_ID': 'count',    'ISSUE_TYPE': 'nunique',}).rename(columns={    'REQUEST_ID': 'TOTAL_REQUESTS',    'ISSUE_TYPE': 'UNIQUE_ISSUE_TYPES'})reqs_agg = reqs_agg.reset_index()print('Service requests aggregation:')print(reqs_agg.head())

In [ ]:
# Combine into master dataframemaster = perms_agg.merge(crime_agg, on=['NEIGHBORHOOD_STANDARD', 'YEAR'], how='outer')master = master.merge(reqs_agg, on=['NEIGHBORHOOD_STANDARD', 'YEAR'], how='outer')print(f'Master dataframe shape: {master.shape}')print(f'Neighborhoods: {master["NEIGHBORHOOD_STANDARD"].nunique()}')print(f'Years: {sorted(master["YEAR"].unique())}')print('\nFirst few rows:')master.head()

## Calculating Rates & Indices

In [ ]:
# Crime rate per 1,000 residents# First, estimate or get population by neighborhood# Option 1: Get from Census API (requires setup)# Option 2: Use manual mappingneighborhood_population = {    'Downtown': 12000,    'North End': 8500,    'Como Park': 9200,    # Add your neighborhoods}# Add to mastermaster['POPULATION'] = master['NEIGHBORHOOD_STANDARD'].map(neighborhood_population)# Calculate crime ratemaster['CRIME_RATE_PER_1000'] = (master['TOTAL_CRIMES'] / master['POPULATION']) * 1000print('Crime rates by neighborhood (latest year):')latest = master[master['YEAR'] == master['YEAR'].max()]rates = latest[['NEIGHBORHOOD_STANDARD', 'CRIME_RATE_PER_1000']].sort_values('CRIME_RATE_PER_1000', ascending=False)print(rates)

In [ ]:
# Development metrics per capitamaster['PERMITS_PER_CAPITA'] = master['TOTAL_PERMITS'] / master['POPULATION']master['REQUESTS_PER_CAPITA'] = master['TOTAL_REQUESTS'] / master['POPULATION']master['PERMIT_COST_PER_CAPITA'] = master['TOTAL_PERMIT_COST'] / master['POPULATION']print('Per capita metrics:')print(master[['NEIGHBORHOOD_STANDARD', 'YEAR', 'PERMITS_PER_CAPITA',               'REQUESTS_PER_CAPITA', 'PERMIT_COST_PER_CAPITA']].head(10))

In [ ]:
# Year-over-year growthmaster['PERMITS_YOY_GROWTH'] = master.groupby('NEIGHBORHOOD_STANDARD')['TOTAL_PERMITS'].pct_change()master['CRIME_YOY_GROWTH'] = master.groupby('NEIGHBORHOOD_STANDARD')['TOTAL_CRIMES'].pct_change()master['REQUESTS_YOY_GROWTH'] = master.groupby('NEIGHBORHOOD_STANDARD')['TOTAL_REQUESTS'].pct_change()print('Growth rates (latest year):')growth = latest[['NEIGHBORHOOD_STANDARD', 'PERMITS_YOY_GROWTH',                   'CRIME_YOY_GROWTH', 'REQUESTS_YOY_GROWTH']]print(growth)

## Building Health Indicators

In [ ]:
# Normalize metrics to 0-100 scale using MinMaxScalerfrom sklearn.preprocessing import MinMaxScalerscaler = MinMaxScaler(feature_range=(0, 100))# Normalize crime rate (inverse - lower is better)valid_crime = master['CRIME_RATE_PER_1000'].dropna().values.reshape(-1, 1)scaler.fit(valid_crime)master['CRIME_RATE_SCALED'] = 100 - scaler.transform(master[['CRIME_RATE_PER_1000']])# Normalize permits per capita (direct - higher is better)valid_perms = master['PERMITS_PER_CAPITA'].dropna().values.reshape(-1, 1)scaler.fit(valid_perms)master['PERMITS_PER_CAPITA_SCALED'] = scaler.transform(master[['PERMITS_PER_CAPITA']])print('Scaled metrics:')print(master[['NEIGHBORHOOD_STANDARD', 'CRIME_RATE_SCALED',               'PERMITS_PER_CAPITA_SCALED']].head(10))

In [ ]:
# Create Safety Index# Lower crime = higher safetymaster['SAFETY_INDEX'] = 100 - master['CRIME_RATE_SCALED']# Apply trend bonus/penaltymaster['CRIME_TREND'] = master.groupby('NEIGHBORHOOD_STANDARD')['CRIME_RATE_PER_1000'].pct_change()master['TREND_BONUS'] = master['CRIME_TREND'].apply(    lambda x: -10 if x > 0.05 else (5 if x < -0.05 else 0))master['SAFETY_INDEX'] = master['SAFETY_INDEX'] + master['TREND_BONUS']master['SAFETY_INDEX'] = master['SAFETY_INDEX'].clip(0, 100)print('Safety Index (latest year):')safety = latest[['NEIGHBORHOOD_STANDARD', 'CRIME_RATE_PER_1000', 'SAFETY_INDEX']].sort_values('SAFETY_INDEX', ascending=False)print(safety)

In [ ]:
# Create Development Index (positive indicator)master['DEVELOPMENT_INDEX'] = master['PERMITS_PER_CAPITA_SCALED']# Create Service Need Index (inverse - more requests = more need)valid_reqs = master['REQUESTS_PER_CAPITA'].dropna().values.reshape(-1, 1)scaler.fit(valid_reqs)master['REQUESTS_PER_CAPITA_SCALED'] = scaler.transform(master[['REQUESTS_PER_CAPITA']])master['SERVICE_NEED_INDEX'] = 100 - master['REQUESTS_PER_CAPITA_SCALED']print('Indicator summary (latest year):')indicators = latest[['NEIGHBORHOOD_STANDARD', 'SAFETY_INDEX',                       'DEVELOPMENT_INDEX', 'SERVICE_NEED_INDEX']]print(indicators)

In [ ]:
# Composite Health Scoremaster['HEALTH_SCORE'] = (    master['SAFETY_INDEX'] * 0.40 +    master['DEVELOPMENT_INDEX'] * 0.30 +    master['SERVICE_NEED_INDEX'] * 0.30)# Categorizemaster['HEALTH_CATEGORY'] = pd.cut(    master['HEALTH_SCORE'],    bins=[0, 40, 60, 80, 100],    labels=['Declining', 'Moderate', 'Good', 'Excellent'])print('Health Scores (latest year):')health = latest[['NEIGHBORHOOD_STANDARD', 'HEALTH_SCORE', 'HEALTH_CATEGORY']].sort_values('HEALTH_SCORE', ascending=False)print(health)

## Analysis & Visualization

In [ ]:
# Correlation analysiscorr_cols = ['TOTAL_CRIMES', 'TOTAL_PERMITS', 'TOTAL_REQUESTS', 'HEALTH_SCORE']corr_matrix = master[corr_cols].corr()plt.figure(figsize=(8, 6))sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, cbar_kws={'label': 'Correlation'})plt.title('Correlation Matrix')plt.tight_layout()plt.show()print('Correlations:')print(corr_matrix)

In [ ]:
# Neighborhood health rankinglatest_year = master['YEAR'].max()rankings = master[master['YEAR'] == latest_year].sort_values('HEALTH_SCORE', ascending=False)fig, ax = plt.subplots(figsize=(12, 8))colors = rankings['HEALTH_CATEGORY'].map({    'Excellent': 'green',    'Good': 'blue',    'Moderate': 'orange',    'Declining': 'red'})ax.barh(rankings['NEIGHBORHOOD_STANDARD'], rankings['HEALTH_SCORE'], color=colors)ax.set_xlabel('Health Score')ax.set_title(f'Neighborhood Health Scores ({int(latest_year)})')ax.set_xlim(0, 100)plt.tight_layout()plt.show()

In [ ]:
# Crime trend by neighborhoodtop_5 = master['NEIGHBORHOOD_STANDARD'].unique()[:5]fig, ax = plt.subplots(figsize=(12, 6))for neighborhood in top_5:    data = master[master['NEIGHBORHOOD_STANDARD'] == neighborhood].sort_values('YEAR')    ax.plot(data['YEAR'], data['CRIME_RATE_PER_1000'], marker='o', label=neighborhood)ax.set_xlabel('Year')ax.set_ylabel('Crime Rate (per 1,000 residents)')ax.set_title('Crime Trends by Neighborhood')ax.legend()ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

In [ ]:
# Clustering neighborhoodsfrom sklearn.cluster import KMeansfrom sklearn.preprocessing import StandardScalerfeatures = ['CRIME_RATE_PER_1000', 'PERMITS_PER_CAPITA', 'REQUESTS_PER_CAPITA']X = latest[features].fillna(0)scaler = StandardScaler()X_scaled = scaler.fit_transform(X)kmeans = KMeans(n_clusters=3, random_state=42)latest['CLUSTER'] = kmeans.fit_predict(X_scaled)fig, ax = plt.subplots(figsize=(10, 8))scatter = ax.scatter(    latest['CRIME_RATE_PER_1000'],    latest['PERMITS_PER_CAPITA'],    c=latest['CLUSTER'],    s=200,    cmap='viridis',    alpha=0.6)ax.set_xlabel('Crime Rate (per 1,000)')ax.set_ylabel('Permits per Capita')ax.set_title('Neighborhood Clusters')plt.colorbar(scatter, label='Cluster')plt.tight_layout()plt.show()

## Exporting Results

In [ ]:
# Save master dataframemaster.to_csv('outputs/master_neighborhoods.csv', index=False)print('✓ Saved master_neighborhoods.csv')# Save rankingsrankings.to_csv('outputs/neighborhood_rankings.csv', index=False)print('✓ Saved neighborhood_rankings.csv')# Summary statisticssummary = master.groupby('NEIGHBORHOOD_STANDARD').agg({    'HEALTH_SCORE': ['mean', 'min', 'max'],    'CRIME_RATE_PER_1000': ['mean', 'min', 'max'],    'TOTAL_PERMITS': 'sum',    'TOTAL_REQUESTS': 'sum'}).round(2)summary.to_csv('outputs/neighborhood_summary.csv')print('✓ Saved neighborhood_summary.csv')print('\nExported files to outputs/ folder')